# Enhanced Drug-Surfactant API Workflow Demo

This notebook demonstrates the new enhanced API workflow with authentication, task management, and decorator-based architecture following the ac-dev-lab pattern.

In [ ]:
import pandas as pd
import numpy as np
import enhanced_api_helper_functions as eapi_hf
import requests
import json

# Test API connection
print("Testing API connection...")
eapi_hf.test_api_connection()

## 1. Authentication

The enhanced API includes JWT-based authentication for security.

In [ ]:
# Login and get access token
try:
    token = eapi_hf.login()
    print(f"✅ Successfully logged in")
    print(f"Token (first 50 chars): {token[:50]}...")
except Exception as e:
    print(f"❌ Login failed: {e}")

## 2. Task Management

List and execute tasks using the decorator-based pattern from ac-dev-lab.

In [ ]:
# List available tasks
tasks = eapi_hf.list_available_tasks()
print("Available Tasks:")
for task in tasks:
    print(f"  📋 {task['name']}")
    print(f"     {task['doc'].strip()}")
    print(f"     Parameters: {task['parameters']}")
    print()

## 3. Protocol Generation and Simulation

Generate sample experimental data and create/simulate protocols.

In [ ]:
# Create sample experimental data
sample_data = [
    {
        "trial_index": 0,
        "drug": 25.5,
        "s1": 10, "s2": 5, "s3": 15, "s4": 8, "s5": 12, "s6": 3,
        "s7": 7, "s8": 20, "s9": 1, "s10": 9, "s11": 6, "s12": 4,
        "surfactant_conc": 30,
        "drug_conc": 25
    },
    {
        "trial_index": 1,
        "drug": 18.3,
        "s1": 8, "s2": 12, "s3": 6, "s4": 14, "s5": 9, "s6": 11,
        "s7": 5, "s8": 16, "s9": 3, "s10": 7, "s11": 13, "s12": 2,
        "surfactant_conc": 35,
        "drug_conc": 18
    }
]

# Convert to DataFrame
df_vol = pd.DataFrame(sample_data)
print("Sample experimental data:")
print(df_vol.head())

In [ ]:
# Generate and simulate protocol using the enhanced API
iteration = 5

try:
    protocol_text, sim_result = eapi_hf.generate_and_simulate_protocol(
        df_vol, 
        iteration=iteration,
        plate_well="A1",
        deepplate_well="A1"
    )
    
    print(f"✅ Protocol generated and simulated successfully!")
    print(f"Protocol length: {len(protocol_text)} characters")
    print(f"Simulation log: {sim_result['run_log']}")
    
    # Show first few lines of protocol
    print("\nFirst 500 characters of generated protocol:")
    print(protocol_text[:500] + "...")
    
except Exception as e:
    print(f"❌ Protocol generation/simulation failed: {e}")

## 4. Task-Based Protocol Operations

Use individual tasks for more granular control.

In [ ]:
# Generate protocol using task API
try:
    protocol_text_task = eapi_hf.generate_protocol_via_task(
        data=sample_data,
        iteration=6,
        plate_well="B2",
        deepplate_well="B2"
    )
    
    print(f"✅ Protocol generated via task API")
    print(f"Protocol length: {len(protocol_text_task)} characters")
    
except Exception as e:
    print(f"❌ Task-based protocol generation failed: {e}")

In [ ]:
# Simulate protocol using task API
try:
    sim_result_task = eapi_hf.simulate_protocol_via_task(protocol_text_task)
    
    print(f"✅ Protocol simulated via task API")
    print(f"Success: {sim_result_task['success']}")
    print(f"Run log: {sim_result_task['run_log']}")
    
except Exception as e:
    print(f"❌ Task-based simulation failed: {e}")

## 5. Protocol Execution (Simulation Mode)

Test protocol execution (runs in simulation mode without hardware).

In [ ]:
# Execute protocol via API
try:
    exec_result = eapi_hf.run_protocol_on_robot(
        protocol_text=protocol_text,
        iteration=iteration,
        run_id=f"demo_run_{iteration}"
    )
    
    print(f"✅ Protocol execution completed")
    print(f"Run ID: {exec_result['run_id']}")
    print(f"Status: {exec_result['status']}")
    print(f"Success: {exec_result['success']}")
    
except Exception as e:
    print(f"❌ Protocol execution failed: {e}")

## 6. API Status and Capabilities

Check comprehensive API status and capabilities.

In [ ]:
# Get comprehensive API status
try:
    status = eapi_hf.get_api_status()
    
    print("📊 API Status Report:")
    print(f"   Status: {status['status']}")
    print(f"   Opentrons Available: {status['opentrons_available']}")
    print(f"   Opentrons Version: {status['opentrons_version']}")
    print(f"   API Version: {status['api_version']}")
    print(f"   Capabilities: {', '.join(status['capabilities'])}")
    print(f"   Registered Tasks: {len(status['registered_tasks'])}")
    
    print("\n📋 Registered Tasks:")
    for task in status['registered_tasks']:
        print(f"   • {task}")
    
except Exception as e:
    print(f"❌ Status check failed: {e}")

## 7. Direct API Calls

Examples of making direct API calls for advanced usage.

In [ ]:
# Direct API call example
import os

API_BASE_URL = os.getenv("DRUG_SURFACTANT_API_URL", "http://localhost:8000")

# Get auth headers
headers = eapi_hf.get_auth_headers()

# Make direct API call to execute a task
response = requests.post(
    f"{API_BASE_URL}/tasks/execute",
    headers=headers,
    json={
        "task_name": "generate_protocol_text",
        "parameters": {
            "data": sample_data[:1],  # Just first row
            "iteration": 99,
            "plate_well": "C3",
            "deepplate_well": "C3"
        }
    }
)

if response.status_code == 200:
    result = response.json()
    print(f"✅ Direct API call successful")
    print(f"Protocol generated with {len(result['result'])} characters")
else:
    print(f"❌ Direct API call failed: {response.status_code}")
    print(response.text)

## Summary

The enhanced API provides:

✅ **Authentication**: JWT-based security with user roles  
✅ **Task Management**: Decorator-based task registration (ac-dev-lab pattern)  
✅ **Protocol Operations**: Generate, simulate, and execute protocols  
✅ **Error Handling**: Comprehensive error reporting and logging  
✅ **Flexibility**: Both high-level helper functions and low-level task API  
✅ **Railway Deployment**: Ready for cloud deployment  

This replaces the old SSH/SCP Jupyter notebook workflow with a modern, scalable API architecture.